In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
df=pd.read_csv('Data/df_limpio.csv')
df.columns=[c.split('\n')[0].strip() for c in df.columns]

df_ml=pd.read_csv('Data/df_cluster_desfavorable.csv')
df_ml_r=df_ml[['Nombre','Cluster']].drop_duplicates(subset=['Nombre'])

df=pd.merge(df,df_ml_r,on='Nombre',how='left')

ROJO = "#D32F2F"
NARANJA = "#F57C00"
VERDE = "#388E3C"
AZUL_OSCURO = "#08306B"
mapa_colores = {'1. Riesgo Crítico': ROJO,'2. Alerta (Salvables)': NARANJA,'3. Sanas': VERDE}

In [ ]:
activo_s=df['Total activo'].replace(0,1e-5)
pasivo_s=df['Pasivo líquido'].replace(0,1e-5)

In [ ]:
df['Solvencia']=(df['Fondos propios']/activo_s).clip(-1,1)
df['Liquidez']=(df['Activo circulante']/pasivo_s).clip(0,5)
df['Carga_Deuda']=((df.get('Acreedores a L. P.',0).fillna(0)+df.get('Deudas financieras',0).fillna(0))/activo_s).clip(0,1.5)
df['ROA']=(df['Resultado del Ejercicio']/activo_s*100).clip(-30,30)

In [ ]:
def semaforo_ml(r):
    if pd.isna(r['Cluster']): return '3. Sanas'
    if r.get('Resultado del Ejercicio',0)>0: return '2. Alerta (Salvables)'
    return '1. Riesgo Crítico'

In [ ]:
df['Estado']=df.apply(semaforo_ml,axis=1)

In [ ]:
def mapear_sec(c):
    try:
        z=int(float(c))
        if 10<=z<=33: return 'Industria'
        if 41<=z<=43: return 'Construcción'
        return 'Servicios'
    except: return 'Servicios'

In [ ]:
df['Sector_Macro']=df['sector'].apply(mapear_sec)

df['Tamaño']=df['Número empleados'].apply(lambda x:'1. Micro (<10)' if x<10 else ('2. Pequeña (10-49)' if x<50 else ('3. Mediana (50-249)' if x<250 else '4. Grande (>250)')))

In [ ]:
rosquilla_graf=px.pie(df,names='Estado',hole=0.6,color='Estado',color_discrete_map=mapa_colores)

rosquilla_graf.update_layout( title=dict(text="TEJIDO EMPRESARIAL PAÍS VASCO",x=0.5,xanchor='center'),
    annotations=[dict(text=f"{len(df)}</b><br>Empresas",showarrow=False,font_size=20,x=0.5,y=0.5)],
    showlegend=False
)

rosquilla_graf.update_traces(textinfo='percent',textfont_size=16)
rosquilla_graf.show()

In [ ]:
df['Tamaño']=df['Tamaño'].str.replace(r'^\d+\.\s*','',regex=True)
df_t=df[df['Estado']!='3. Sanas'].copy()

intervencion_graf=px.histogram(
    df_t,x="Tamaño",color="Estado",
    color_discrete_map=mapa_colores,
    barmode="group",text_auto=True,
    title="TARGET DE INTERVENCIÓN POR TAMAÑO"
)

intervencion_graf.update_layout(
    title=dict(x=0.5,xanchor='center'),
    xaxis={'categoryorder':'total descending'},
    xaxis_title="",
    yaxis_title="Número de Empresas",
    height=500,
    showlegend=False
)

intervencion_graf.show()

In [ ]:
df_p = df.groupby(['Provincia', 'Estado']).size().reset_index(name='n')
df_p['pct'] = df_p['n'] / df_p.groupby('Provincia')['n'].transform('sum') * 100

semaforo_graf=px.bar(
    df_p,
    x="pct",
    y="Provincia",
    color="Estado",
    color_discrete_map=mapa_colores,
    orientation='h',
    text=df_p['pct'].map('{:.1f}%'.format),
    category_orders={'Estado': ['3. Sanas', '2. Alerta (Salvables)', '1. Riesgo Crítico']},
    title="SEMÁFORO TERRITORIAL"
)

semaforo_graf.update_layout( xaxis_title="%", showlegend=False)
semaforo_graf.update_traces(textposition='inside')
semaforo_graf.show()

In [ ]:
geo = {'BILBAO':(43.26,-2.93),'VITORIA-GASTEIZ':(42.84,-2.67),'DONOSTIA':(43.31,-1.98),'DONOSTIA/SAN SEBASTIAN':(43.31,-1.98),'PAMPLONA/IRUÑA':(42.81,-1.64),'TUDELA':(42.06,-1.60),'ZAMUDIO':(43.28,-2.86)}

prov={'Gipuzkoa':(43.15,-2.1),'Bizkaia':(43.25,-2.8),'Araba':(42.85,-2.6),'Navarra':(42.6,-1.6)}

df_m = df[df['Estado']=='2. Alerta (Salvables)'].copy()

coords = df_m['Localidad'].str.upper().map(geo)
prov_coords = df_m['Provincia'].map(prov)

df_m['lat'] = coords.str[0].fillna(prov_coords.str[0])
df_m['lon'] = coords.str[1].fillna(prov_coords.str[1])

mapa_graf = px.density_mapbox(
    df_m, lat='lat', lon='lon', z='Total activo', radius=25,
    center=dict(lat=43.05, lon=-2.2), zoom=7.5,
    mapbox_style='carto-positron',
    color_continuous_scale=[[0,"rgba(255,255,255,0)"],[1,mapa_colores['2. Alerta (Salvables)']]]
)

mapa_graf.update_layout(
    title=dict(text="FOCOS DE INTERVENCIÓN: CUENTAS SALVABLES", x=0.5),
    coloraxis_showscale=False, margin=dict(l=0,r=0,t=60,b=0), height=600
)

mapa_graf.show()

In [ ]:
df_salvables = df[df['Estado']=='2. Alerta (Salvables)']

print(f"Total empresas Salvables en el mapa: {len(df_salvables)}\n")

print("--- TOP 5 FOCOS (Por CANTIDAD de empresas) ---")
print(df_salvables['Localidad'].value_counts().head(5))

print("\n--- TOP 5 FOCOS (Por MILLONES DE EUROS de deuda) ---")
print((df_salvables.groupby('Localidad')['Deudas financieras'].sum()/1000)
      .sort_values(ascending=False).head(5).round(2))

In [ ]:
df_per = df[(df['Estado'] != '3. Sanas') & (df['Resultado del Ejercicio'] < 0)].copy()
df_per['Meses_Vida'] = (df_per['Tesorería'] / (abs(df_per['Resultado del Ejercicio']) / 12)).clip(0, 24)
df_per['Tramo'] = pd.cut( df_per['Meses_Vida'], bins=[-1, 3, 6, 24], labels=['Crítico (< 3 meses)', 'Intervención (3-6meses)', 'Corto Plazo (> 6meses)'])


df_f = (
    df_per['Tramo']
    .value_counts()
    .reset_index(name='Empresas')
    .rename(columns={'index': 'Tramo'})
    .sort_values('Tramo', ascending=False)
)

urgencia_comercial_graf = px.funnel(
    df_f,
    x='Empresas',
    y='Tramo',
    color='Tramo',
    color_discrete_map={
        'Crítico (< 3 meses)': "#900C3F",
        'Intervención (3-6meses)': "#C70039",
        'Corto Plazo (> 6meses)': "#FFC300"
    },
    title="Urgencia Comercial: Meses de Oxígeno en Caja"
)

urgencia_comercial_graf.update_layout(template="plotly_white", margin=dict(l=250), showlegend=False)
urgencia_comercial_graf.show()

In [ ]:
df_v = df[(df['ROA'] >= -10) & (df['ROA'] <= 10)].copy()

boxplot_graf = px.box(
    df_v,
    x="Sector_Macro",
    y="ROA",
    color="Sector_Macro",
    points=False,
    color_discrete_sequence=["#1F618D", "#7D3C98", "#5D6D7E"],
    title="Rentabilidad por Sector"
)

boxplot_graf.add_hline( y=0, line_dash="dot", line_color="#E74C3C")
boxplot_graf.update_layout( template="plotly_white", showlegend=False)
boxplot_graf.show()

In [ ]:
solvencia_graf=px.violin(
    df,
    y="Solvencia",
    color="Estado",
    box=True,
    color_discrete_map=mapa_colores,
    title="Distribución de Solvencia"
)

solvencia_graf.update_layout(template="plotly_white",plot_bgcolor="white",paper_bgcolor="white")
solvencia_graf.show()

In [ ]:
df_plot_liq=df[
    (df['Estado'].isin(['3. Sanas', '2. Alerta (Salvables)'])) &
    (df['Liquidez'] >= 0) &
    (df['Liquidez'] <= 4)
].copy()

liq_graf = px.histogram(
    df_plot_liq,
    x="Liquidez",
    color="Estado",
    marginal="box",
    facet_row="Estado",
    nbins=50,
    color_discrete_map=mapa_colores,
    title="Análisis de Distribución: Ratio de Liquidez</b><br><sup>Comparativa en carriles separados (Sanas vs Alerta)</sup>"
)

liq_graf.update_layout(
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=500,
    showlegend=False,
    xaxis_title="Ratio de Liquidez a Corto Plazo (Peligro < 1)"
)

liq_graf.add_vline(x=1, line_dash="dash", line_color="red")
liq_graf.show()

In [ ]:
sns.set_theme(style="white")  

colores_sns = {'1. Riesgo Crítico': ROJO,'2. Alerta (Salvables)': NARANJA,'3. Sanas': VERDE}
orden_semaforo = [ '1. Riesgo Crítico', '2. Alerta (Salvables)', '3. Sanas']

In [ ]:
plt.figure(figsize=(10,5))
sns.violinplot(
    data=df,
    x='Estado',
    y='Solvencia',
    palette=colores_sns,
    inner='quartile',
    order=orden_semaforo
)

plt.title('Estadística Descriptiva: Solvencia Patrimonial',fontsize=14,pad=15)

plt.ylabel('Ratio de Solvencia')
plt.xlabel('')
plt.axhline(0,color='black',linestyle='--',linewidth=1.5,label='Quiebra Técnica (Patrimonio < 0)')

plt.legend()
sns.despine()  
plt.show()

In [ ]:
variables_estudio = ['Liquidez', 'ROA', 'Solvencia', 'Carga_Deuda']
f_stats = {}

for var in variables_estudio:
    g1 = df[df['Estado'] == '1. Riesgo Crítico'][var].dropna()
    g2 = df[df['Estado'] == '2. Alerta (Salvables)'][var].dropna()
    g3 = df[df['Estado'] == '3. Sanas'][var].dropna()
    
    f, p = stats.f_oneway(g1, g2, g3)
    f_stats[var] = f

In [ ]:
df_anova = pd.DataFrame({
    'Variable': [
        'Liquidez a C.P.',
        'Rentabilidad (ROA)',
        'Solvencia (Patrimonio)',
        'Carga de Deuda'
    ],
    'F_Statistic': [
        f_stats['Liquidez'],
        f_stats['ROA'],
        f_stats['Solvencia'],
        f_stats['Carga_Deuda']
    ]
}).sort_values('F_Statistic', ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(data=df_anova,x='F_Statistic',y='Variable',color=AZUL_OSCURO)

plt.title( 'Inferencia: Las variables que mejor separan los clústeres', fontsize=14,fontweight='bold',pad=15)
plt.ylabel('')
plt.show()